# Base learners

Base learners are the regression models fitted to the negative gradient at
each boosting iteration; the engine selects the learner that best fits the
pseudo-residual and adds a fraction $\nu$ of its prediction to the current
model ([](functional_gradient_boosting.ipynb)). The types available in
`mboostrs` are linear (`bols`), P-spline (`bbs`), ridge-penalized B-spline
(`bns`), monotone (`bmono`), constrained (`bconstrained`), tree (`btree`),
random effect (`brandom`), spatial (`bspatial`), Markov random field (`bmrf`),
kernel (`bkernel`) and user-defined (`buser`). The `df` calibration that
equalizes learners is in [](penalties_df.ipynb); [](../usage/glmboost) and
[](../usage/gamboost) have runnable examples.

## The linear smoother interface

Most base learners in `mboostrs` are **penalized linear smoothers**. In matrix
form, the fitted values are

$$
\hat u = H(\lambda) u = X (X^\top W X + \lambda K)^{-1} X^\top W u,
$$

where $u$ is the pseudo-residual (negative gradient), $X$ is the design, $W$
the observation weights, $K$ the penalty matrix, and $\lambda$ the smoothing
parameter. The learner is fully specified by $(X, K, W, \lambda)$; the engine
only needs the hat matrix $H$ and its interaction with $u$. The exceptions
are `btree`, whose split points depend on $u$, and `bmono` and `bconstrained`
when a constraint is active, whose fit is the solution of a constrained
problem; for these three the fitted values are not a fixed linear map of $u$.

The $\lambda$ parameter is usually not set directly. Instead the user
specifies a target **degrees of freedom** $\mathrm{df}^\star$ and the engine
solves $\mathrm{df}(\lambda) = \mathrm{df}^\star$ for $\lambda$ (`lambda_value=`
sets it directly). Learners given the same $\mathrm{df}^\star$ are equally
flexible, so that selection reflects signal rather than flexibility. The
defaults are `df=4` for `bbs`, `bmono`, `bconstrained`, `brandom`, `bmrf` and
`bkernel`, `df=6` for `bspatial`, and no penalty for `bols`
(see [](penalties_df.ipynb) and [](from_glm_to_boosting.ipynb) §3.3).



In [1]:
import numpy as np

n, p = 10, 4
rng = np.random.default_rng(42)
X = rng.standard_normal((n, p))
w = np.ones(n)
λ = 2.0

# Ridge penalty K = I
K = np.eye(p)

# Penalized Gram matrix G = XᵀWX + λK
G = X.T @ np.diag(w) @ X + λ * K

# Hat matrix H(λ) = X (XᵀWX + λK)⁻¹ XᵀW
H = X @ np.linalg.solve(G, X.T * w)   # (n×p) @ (p×n) = n×n

df_mallows = np.trace(2 * H - H @ H.T)
df_simple = np.trace(H)
print(f"tr(H)           = {df_simple:.4f}")
print(f"tr(2H - HHᵀ)    = {df_mallows:.4f}")
print(f"rank(X)         = {np.linalg.matrix_rank(X)}")

# As λ → ∞, df → dim ker K (= 0 for ridge with K=I)
G_inf = X.T @ np.diag(w) @ X + 1e12 * K
H_inf = X @ np.linalg.solve(G_inf, X.T * w)
print(f"df(λ→∞) ≈ {np.trace(2*H_inf - H_inf @ H_inf.T):.6f}  (should → 0)")


tr(H)           = 2.6915
tr(2H - HHᵀ)    = 3.3951
rank(X)         = 4
df(λ→∞) ≈ 0.000000  (should → 0)


## Learner catalog

### `bols` — linear effect

Ordinary least squares on one or more columns. By default (`df=None`) `bols`
is unpenalized; giving `df=` or `lambda_value=` adds a ridge penalty
($K = I$) that shrinks the coefficients toward zero. With `intercept=True`
(the default), the intercept is excluded from the penalty.

In `glmboost`, each `bols(x)` is one centered column, so its fit is a scalar
division. In `gamboost`, a multi-column `bols` is solved with a Cholesky
factorization of its Gram matrix.

Use `bols` when you want a **linear** effect (possibly with an intercept-only
grouping factor). Do not use it for high-cardinality factors; use `brandom`
instead (see [](#categorical-predictors-and-brandom) below).

### `bbs` — P-spline (penalized B-spline)

The standard smooth learner. It constructs a B-spline basis of the given degree
(default 3, cubic) with the specified number of inner knots (default 20), then
penalizes the $d$-th order differences of the spline coefficients (default
$d = 2$, penalising deviations from linearity).

The difference penalty of order $d$ has a null space of dimension $d$: a
second-order penalty leaves constant and linear functions unpenalized. This
means a `bbs` learner with the default settings shrinks the fitted curve
*toward a straight line* — the penalty's null space. As $\lambda$ increases,
the effective degrees of freedom asymptotes to $d = 2$, never below.

The design matrix has far more columns than `df`. The basis is always
`knots + degree + 1` columns wide (24 for the defaults), and `df` is the
number of effective parameters the penalty leaves after shrinkage — a
smoothness parameter, not a column count. A small `df` is a strong penalty, not a
small basis. A `df` below the null-space rank (with `center=False`) raises an
error when the learner is built, and a `df` equal to it gives a warning and
the maximum penalty; `center=True` removes the null space from the design,
which is how `df=1` becomes attainable — see
[](penalties_df.ipynb) for the calibration.

### `bns` — ridge-penalised B-spline

A B-spline basis like that of `bbs`, with an identity (ridge) penalty in
place of the difference penalty. By default (`df=None`) the basis is fitted
effectively unpenalized; a smaller `df` adds the ridge penalty. The ridge
penalty has no null space, so it shrinks all coefficients, including the
constant and linear components, toward zero. The R package `mboost` does not provide `bns`. Use it when the
fit should not be anchored to a line, as `bbs` is.

### `bmono` — monotone/shape-constrained P-spline

A `bbs` with a constraint that the fitted function satisfies a monotonicity or
shape restriction (increasing, decreasing, convex, concave, positive, or
negative). Two solvers are available:

1. **`quad.prog`** (default, as in R's `bmono`): a primal active-set quadratic
   program that solves the constrained problem *exactly*, so the coefficients
   are strictly feasible rather than approximately so. Matches `mboost` to
   about 5e-10.
2. **`iterative`**: a violation-penalty fixed point that penalizes constraint
   violations with a large $\lambda_2$. Roughly 5–10 times faster per step
   and usually sufficient for the boosting inner loop.

### `bconstrained` — general linear-constraint P-spline

Generalizes `bmono` to an arbitrary constraint matrix $C$ ($m \times p$) such
that $C\beta \ge 0$. The user supplies $C$ directly; for monotone constraints
$C$ is the first-difference matrix, for convexity $C$ is the second-difference
matrix. Each step is solved with the same two solvers as `bmono`: exactly by
the active-set quadratic program (`type="quad.prog"`, the default) or approximately by the
violation penalty (`type="iterative"`).

(spectral-centering)=

## Spectral centering: `bbs(center=TRUE)` and `bmrf(center=TRUE)`

A second-order difference penalty leaves a 2-dimensional null space (constant
and linear functions). The penalty can shrink the *wiggly* part of the spline
toward zero, but it cannot shrink the null-space components — those are always
fit unpenalized.

Spectral centering removes the null space before fitting. The penalty matrix
$K$ is eigendecomposed: $K = V D V^\top$, where $D = \operatorname{diag}(d_1,
\ldots, d_p)$ with $d_\ell = 0$ for null-space directions and $d_\ell > 0$
otherwise. The centered design is

$$
X_c = X \, V_{>0} \, \operatorname{diag}(d_{>0})^{-1/2},
$$

where $V_{>0}$ are the eigenvectors with positive eigenvalues. After this
transformation the penalty becomes the identity $K_c = I$, and the null space
is eliminated (the column count drops from $p$ to $p - d$).

For a default `bbs` ($d = 2$ differences), this reduces the coefficient
vector from $p = n_{\text{knots}} + \text{degree} + 1 = 24$ columns to
$p - 2 = 22$.

### Effects of centering

| Property | `center=FALSE` (default) | `center=TRUE` |
|---|---|---|
| Shrinkage target | Linear function (penalty null space) | Zero ($K_c = I$ penalizes all directions) |
| Minimum attainable $\mathrm{df}$ | $d$ (the null-space dimension) | 0 (identity penalty has no null space) |
| Intercept handling | Linear trend absorbed by the spline | Constant + linear absorbed by the model intercept |
| Number of coefficients | $p$ | $p - d$ |

When `center=TRUE`, the spline is not anchored to a polynomial. With
small `df`, a centered spline shrinks toward zero rather than toward the best
linear fit. This is appropriate when the intercept and any linear trend should
be estimated by separate linear base learners rather than by the smooth term
itself.

In `mboostrs`, `center` defaults to `False` for every learner, including
`bspatial`.

### Uses of `center=TRUE`

- When you want the smooth term to capture *only* the nonlinear component,
  with the linear trend picked up by a separate `bols(x)` learner.
- When you want the minimum $\mathrm{df}$ to be below the null-space
  dimension (e.g., to strongly regularize a spatial effect).
- In spatial models (`bspatial`), where the constant surface should be absorbed
  into the model intercept.

### Uses of `center=FALSE` (the default)

- When you want the smooth to capture both the overall trend and the
  nonlinear deviation (the typical GAM setup).
- When you are comparing `bbs` learners with `bols` learners at the same
  `df` — the null-space anchoring makes the comparison fair because the
  `bbs` learner cannot win simply by having a lower minimum df.

### Prediction on new data with `center=TRUE`

For `center=TRUE` fits, prediction on *new* data requires rebuilding the
centered B-spline design via the transformation matrix $C = V_{>0}
\operatorname{diag}(d_{>0})^{-1/2}$, so that $X_{\text{new},c} = X_{\text{new}} C$.
In Python, `predict(newdata=...)` applies this transformation for centered
fits. In the R interface, prediction on new data rebuilds the uncentered design, so predictions from centered fits can be wrong without an error.

### `bmrf` centering

`bmrf(center=TRUE)` works identically: the neighborhood matrix $K$ is
eigendecomposed, the null space (usually one dimension — the constant, since
$K$ has zero row sums) is dropped, and the penalty becomes the identity.
This is the spatial analogue: the spatially-constant effect is absorbed into
the intercept.

(categorical-predictors-and-brandom)=

## Categorical predictors and `brandom`

A factor with $L$ levels, passed to `bols`, creates an $n \times L$ indicator
design. With a ridge penalty, this is the fixed-effects encoding. For a
small number of levels (say $L < 10$), `bols` is fine.

For **high-cardinality** factors (hundreds or thousands of levels — e.g.,
postal code, provider ID), `bols` has two problems:

1. The indicator matrix has one column per level, so the per-learner `df`
   budget is spread across all of them and the fit is noisy.
2. Each level estimates a separate fixed effect; when levels share
   information (nearby regions, similar providers), a random-effects model
   that borrows strength across levels is more efficient.

`brandom` fits the factor as a random effect: the indicator design is the
same, but the ridge penalty shrinks all level effects toward their weighted
mean (equivalently, toward the intercept). The amount of shrinkage is
controlled by `df`. In the boosting framework, `brandom` gives smooth
selection dynamics: levels with strong signal stand out, weak levels shrink
toward zero, and the overall factor can be effectively turned off by not
selecting it.

**Rule of thumb:** use `brandom` instead of `bols` for any factor with more
than about 10 levels. For spatial effects on a factor with a known
neighborhood graph, use `bmrf`. The automatic specification search applies
its own rule for choosing between them; [](../usage/choosing_terms.ipynb)
states it.

## Varying-coefficient effects: the `by` argument

Every linear learner accepts a `by=` variable that multiplies the design by a
observation-specific scalar:

$$
X_{\text{by}}[i, j] = z_i \cdot X[i, j],
$$

where $z$ is the `by` variable. This implements a **varying-coefficient**
effect: the smooth shape of `bbs(x)` is scaled by $z$. When $z \in \{0, 1\}$,
this selects a subgroup; when $z$ is continuous, the smooth effect of $x$
varies proportionally to $z$.

With $x$ equal to the observed event or censoring time, this algebra gives a
coefficient that varies with that observed time across subjects. A
time-dependent Cox effect, whose value changes within each subject's follow-up,
requires start-stop risk sets, which the survival engine does not implement.

A factor `by` variable has no scalar $z_i$ to multiply by. It is coded with
treatment contrasts instead — indicator columns for every level but the first —
and the design is stacked once per remaining level:

$$
X_{\text{by}} = \left[\, d_2 \odot X, \;\; d_3 \odot X, \;\; \cdots, \;\;
d_k \odot X \,\right],
\qquad
K_{\text{by}} = I_{k-1} \otimes K,
$$

where $d_l$ is the indicator of level $l$ and $\odot$ multiplies it into every
column. The first level is the reference: the term contributes nothing for it,
and each other level fits its difference from it. The penalty is block
diagonal, so the blocks are shrunk to the same degree by one $\lambda$, and the
term's `df` covers all of them — which is why it cannot go below
$d\,(k-1)$ for a difference penalty of order $d$, the null space having stacked
along with the design.

## Other learner types

### `bmrf` — Markov random field

For a categorical covariate with a spatial or relational structure. The
design is an indicator matrix (one column per level) and the penalty $K$ is the
graph Laplacian of the neighborhood structure, supplied by the caller:
$K_{ii}$ is the number of neighbors of region $i$, $K_{ij} = -1$ when $i$
and $j$ are adjacent, and 0 otherwise. Then
$\beta^\top K \beta = \sum_{i \sim j} (\beta_i - \beta_j)^2$, summed over
adjacent pairs, so the penalty grows as neighboring effects diverge and
adjacent levels are pulled toward each other.

The adjacency matrix — zero diagonal, $+1$ for neighbors — is a different
matrix and will not work here. It is indefinite, so $\beta^\top K \beta$ can
be negative and the penalty rewards exactly the variation it is meant to
suppress. `mboostrs` rejects it, as does `mboost`.

### `bspatial` — tensor-product P-spline

A 2-D smooth surface constructed from the Kronecker product of two marginal
B-spline bases. The penalty is the additive Kronecker penalty
$K_1 \otimes I_2 + I_1 \otimes K_2$. Defaults to `center=False`.

### `btree` — decision tree

Fits a shallow decision tree (default: depth 1, a stump, which is a single
split) to the pseudo-residual. The split points depend on the pseudo-residual,
so the fit is not a fixed linear map of it, and `btree` takes no `df`
argument. Given the fitted partition, a step is a projection onto the region
indicators; `aic()` uses that conditional hat matrix, so its df exclude the
search over splits. Useful for capturing discontinuities
or interactions that splines cannot represent.

### `bkernel` — kernel ridge regression

Constructs an $n \times n$ kernel matrix $K[i,j] = k(\|x_i - x_j\| / h)$ and
fits a kernel ridge regression. Memory is $O(n^2)$, so this is practical only
for moderate $n$ (up to a few thousand).

The Nyström method approximates a large kernel matrix from its columns at a
subset of $m$ points, the landmarks. The Nyström alternative replaces the
$n \times n$ design with an $n \times m$ one against $m$ evenly spaced quantile
landmarks of $x$. With $K_{nm}$ the
kernel between observations and landmarks and $K_{mm} = L L'$ the landmark
kernel, the design is $\Phi = K_{nm} L^{-\intercal}$, so
$\Phi\Phi' = K_{nm} K_{mm}^{-1} K_{mn}$, the Nyström approximation of
$K_{nn}$. Only a triangular solve with the factor $L$ (a linear system whose matrix is triangular, so it is solved by substitution) is needed, memory drops to $O(n \cdot m)$,
and the approximation is exact as $m$ approaches the number of distinct
observations. The compact-support kernels (Epanechnikov, uniform) can make
the landmark matrix singular, so the approximation is restricted to the
Gaussian kernel.

### `buser` — user-supplied design and penalty

Accepts raw arrays $X$ (design) and $K$ (penalty) and wraps them as a
multi-column linear learner. The most flexible learner; users can define arbitrary penalized
linear structures without writing Rust code. Shape validation is enforced at
construction: $X$ must be 2-D, $K$ must be square, and $X$ columns must match
$K$ dimension.

### Mathematical form of difference penalties

For a P-spline (`bbs`), the penalty matrix $K$ enforces smoothness by targeting
the $d$-th order differences of adjacent basis coefficients:

$$ K = D_d^\top D_d $$

where $D_d$ is the $d$-th difference matrix of size $(p - d) \times p$ and $p$
is the basis dimension. For example, when $p = 4$:

- **First-difference operator** ($d = 1$, penalizing deviations from a
  constant):

$$ D_1 = \begin{pmatrix} -1 & 1 & 0 & 0 \\ 0 & -1 & 1 & 0 \\ 0 & 0 & -1 & 1 \end{pmatrix} $$

- **Second-difference operator** ($d = 2$, penalizing deviations from a
  straight line):

$$ D_2 = \begin{pmatrix} 1 & -2 & 1 & 0 \\ 0 & 1 & -2 & 1 \end{pmatrix} $$

The null space of $D_d$ contains all polynomials of degree $d - 1$. For the
default $d = 2$, constant and linear trends are completely unpenalized,
allowing the spline to capture overall slopes freely while regularizing
non-linear curvature.

### Optimization in shape-constrained solvers

For `bmono` and `bconstrained`, the coefficients $\beta$ are subjected to
inequality constraints $C\beta \ge 0$. The iterative solver minimizes the
following penalized objective at each boosting step:

$$ \mathcal{Q}(\beta) = \sum_{i=1}^n w_i (u_i - [X\beta]_i)^2 + \lambda_1 \beta^\top K_1 \beta + \lambda_2 \sum_{k} \max(0,\; -C_k\beta)^2 $$

where:

- $u_i$ is the pseudo-residual for observation $i$
- $K_1$ is the standard difference penalty matrix (e.g. $D_2^\top D_2$)
- $C_k$ is the $k$-th row of the constraint matrix
- $\lambda_2$ is a dynamic penalty parameter

The solver iteratively increases $\lambda_2$ for violated constraints
($C_k\beta < 0$) and re-solves the KKT system until all violations fall below
a tolerance ($10^{-6}$), producing a feasible fit. The default `quad.prog`
solver is a primal active-set quadratic program that guarantees strict
feasibility and is slower.



In [2]:
import numpy as np

# Build D₁ (first-difference) and D₂ (second-difference) matrices
p = 6

D1 = np.zeros((p - 1, p))
for i in range(p - 1):
    D1[i, i:i+2] = [-1, 1]

D2 = np.zeros((p - 2, p))
for i in range(p - 2):
    D2[i, i:i+3] = [1, -2, 1]

print("D₁ (first difference):")
print(D1.astype(int))
print(f"\nD₂ (second difference):")
print(D2.astype(int))

null1 = p - np.linalg.matrix_rank(D1)
null2 = p - np.linalg.matrix_rank(D2)
print(f"\nNull-space dimension: D₁={null1} (constant), D₂={null2} (linear)")

β = np.arange(p, dtype=float)
print(f"K₁ penalization of linear β: {β @ D1.T @ D1 @ β:.4f}")
print(f"K₂ penalization of linear β: {β @ D2.T @ D2 @ β:.4f}  (should be 0)")


D₁ (first difference):
[[-1  1  0  0  0  0]
 [ 0 -1  1  0  0  0]
 [ 0  0 -1  1  0  0]
 [ 0  0  0 -1  1  0]
 [ 0  0  0  0 -1  1]]

D₂ (second difference):
[[ 1 -2  1  0  0  0]
 [ 0  1 -2  1  0  0]
 [ 0  0  1 -2  1  0]
 [ 0  0  0  1 -2  1]]

Null-space dimension: D₁=1 (constant), D₂=2 (linear)
K₁ penalization of linear β: 5.0000
K₂ penalization of linear β: 0.0000  (should be 0)


### Numerical check: partition of unity and null space

Two properties of `bbs` can be checked on the basis construction and a fit:
the B-spline basis forms a partition of unity with local support, and a pure
linear function passes through a `bbs(differences=2)`
learner unpenalized: a second-difference penalty has the linear functions in
its null space, so their penalty contribution is exactly zero.


In [3]:
import numpy as np
import mboostrs as mb
import mboostrs._rust as _rust

# Property 1: partition of unity + local support, from the actual basis
# construction used by bbs (not a hand-rolled B-spline implementation).
x_grid = np.linspace(0, 10, 200)
flat_design, p_basis = _rust.bbs_raw_design(x_grid, (0.0, 10.0), 20, 3)
design = np.asarray(flat_design).reshape(len(x_grid), p_basis)
row_sums = design.sum(axis=1)
nonzero_per_row = (design != 0).sum(axis=1)
print(f"basis columns: {p_basis}")
print(f"partition of unity: row sums in [{row_sums.min():.10f}, {row_sums.max():.10f}]")
print(f"local support: {nonzero_per_row.max()} of {p_basis} basis functions "
      f"nonzero at any point (degree+1 = 4)")
assert np.allclose(row_sums, 1.0)
assert nonzero_per_row.max() <= 4

# Property 2: a pure linear function is (almost) unpenalized by a
# second-order difference penalty -- fit bbs(differences=2) to noiseless-ish
# linear data and check the fitted curve recovers the line, not a shrunk
# version of it.
rng = np.random.default_rng(0)
n = 300
x = np.sort(rng.uniform(0, 10, n))
y_true = 2.0 + 1.5 * x
y = y_true + rng.normal(size=n) * 0.01
fit = mb.gamboost("y ~ bbs(x, df=4, differences=2)", data={"y": y, "x": x},
                  family=mb.Gaussian(), control=mb.boost_control(mstop=200, nu=0.3))
max_dev = np.abs(fit.fitted_ - y_true).max()
print(f"\nmax |fitted - true linear| = {max_dev:.4f} (noise std was 0.01)")
assert max_dev < 0.05


basis columns: 24
partition of unity: row sums in [1.0000000000, 1.0000000000]
local support: 4 of 24 basis functions nonzero at any point (degree+1 = 4)

max |fitted - true linear| = 0.0026 (noise std was 0.01)


### Behaviour outside the fitted range

A B-spline basis is defined between its boundary knots $[a, b]$. For
$x \notin [a, b]$ the basis functions are evaluated by linear extension, so the
fitted effect continues with the slope it had at the edge. Nothing in the
penalty or the data constrains that continuation: the second-difference
penalty acts on the coefficients, and outside the knot range no coefficient is
identified by any observation.

Two consequences follow. The extrapolated effect is a straight line whatever
the interior shape was, so a curve that turned near the boundary keeps going
in whichever direction it happened to be heading. And the further out the
evaluation point, the more the prediction depends on the last interior
coefficient alone.

`predict` reports the distance in training ranges, and `extrapolation="clip"`
replaces the linear continuation with a flat one by evaluating the basis at
the nearest boundary knot. Linear continuation assumes the edge slope persists;
clipping assumes the effect remains constant beyond the fitted range.

`bols` is unaffected: a linear learner is defined on the whole line, and
predicting outside the training range continues the estimated line rather than
an unconstrained basis.


## Joint multivariate learners

R `mboost` allows a single base learner to span multiple covariates, as in
`bols(x1, x2)` with a joint design. The `mboostrs` formula parser does not accept
that spelling. It provides interactions through the `%X%` operator instead, which
composes two single-variable learners into one candidate.

### The `%X%` tensor interaction

`bbs(x) %X% brandom(g)` builds a learner whose design is the row-wise tensor
product of the two marginal designs. For marginals $X_L \in \mathbb{R}^{n \times
p_L}$ and $X_R \in \mathbb{R}^{n \times p_R}$, row $i$ of the interaction design
is the outer product of the two marginal rows, flattened:

$$
X_{[i, \cdot]} = \operatorname{vec}\!\left( X_{L[i,\cdot]} \, X_{R[i,\cdot]}^\top \right)
\in \mathbb{R}^{p_L p_R}.
$$

This is the Khatri–Rao (row-wise Kronecker) product, not the full Kronecker
product: it pairs each row with itself rather than forming all $n^2$ row
combinations, so the design stays $n \times p_L p_R$.

The penalty is the **Kronecker sum** of the marginal penalties,

$$
K = K_L \otimes I_{p_R} + I_{p_L} \otimes K_R,
$$

which is the standard anisotropic tensor-product penalty
{cite}`eilers1996flexible`. Its meaning is additive roughness: the first term
penalizes variation along the left marginal at every level of the right, the
second penalizes variation along the right at every level of the left. A single
$\lambda$ then scales both, so the term carries one smoothing parameter and one
`df`, and is selected or rejected as one unit like any other learner.

That single `df` is the **product** of the two marginal `df`s, which is R
`mboost`'s rule for the same operator: `bbs(x, df=4) %X% bbs(z, df=4)` calibrates
$\lambda$ to 16 degrees of freedom, and a margin carrying no `df` of its own —
`bols`, which is unpenalized — counts as 1. Writing `lambda=` on a margin instead
multiplies it into that margin's block of the Kronecker sum, so the two
directions can be given different smoothness after all; when both margins do it,
the tensor takes $\lambda = 1$ and no `df`, there being nothing left to
calibrate.

The null space of $K$ is the tensor product of the two marginal null spaces, so
its dimension is their product and one unpenalized margin is not enough to make
it non-trivial — both must leave something free. For `bbs(x) %X% brandom(g)` the
ridge penalty of `brandom` leaves nothing free, so $K$ is positive definite and
the interaction shrinks toward zero. For `bbs(x) %X% bols(g)`, where `bols`
contributes no penalty at all, the free directions are the two linear functions
of $x$ at each of the two `bols` columns: four, and the term shrinks toward a
per-group linear trend rather than toward zero.

That dimension is a floor on the attainable `df`: $\mathrm{df}(\lambda)$ falls
toward it as $\lambda$ grows and never reaches it, so marginal `df`s multiplying
to 4 or less on a two-`bbs` tensor ask for a shrinkage no $\lambda$ delivers.
The learner warns and is fitted at maximum shrinkage.

Because one penalty and one `df` cover both directions, `%X%` cannot express
different smoothness along each margin from a `df` alone. A term needing that —
much rougher in one coordinate than the other — wants `bspatial`, which carries a
separate $\lambda$ per marginal basis, or an explicit `lambda=` on each margin.

The composed design and penalty enter the engine as an ordinary multi-column
learner. [](../usage/real_data.ipynb) has a worked comparison against the
additive fit.

## Summary

- Each base learner is a regression tool with a design matrix $X_j$ and an
  optional penalty $K_j$. The `df` parameter sets the effective degrees of
  freedom, and the engine solves for $\lambda$ via the generalized eigenvalue
  problem $(K, X^\top W X)$.
- Smooth learners (`bbs`, `bspatial`) use B-spline bases with difference
  penalties; `bns` uses a B-spline basis with a ridge penalty;
  shape-restricted learners (`bmono`, `bconstrained`) add a constraint matrix
  $C$ to the solver. The default `df=4` of `bbs` is robust for most problems.
- Learners with matrix arguments (`bmrf`'s $K$, `bconstrained`'s $C$,
  `buser`'s $X/K$) cannot round-trip through a formula string — supply them
  programmatically.

## See also

- [](penalties_df.ipynb) — the `df`-to-$\lambda$ mapping in detail.
- [](../usage/base_learners) — runnable examples for every learner type.
- [](../implementation/base_learners) — the Rust implementation of each
  learner.